# 03 - Ablation Study (Multi-Config)
## Feature reduction: Full → Top-30 → Top-20 → Top-10 → Top-5

Input: `cleaned_train_*K.pkl`, `feature_importance_*K.csv`
Output: `ablation_results_all.pkl`

In [ ]:
import pandas as pd
import numpy as np
import pickle, time, os, gc, tempfile
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

CONFIGS = [10000, 30000, 50000, 100000]
FEATURE_CONFIGS = [('Full', None), ('Top-30', 30), ('Top-20', 20), ('Top-10', 10), ('Top-5', 5)]
DATA_DIR = '../data/'
all_ablation = {}
print('OK')

In [ ]:
for config in CONFIGS:
    suffix = f'{config//1000}K'
    pkl_path = os.path.join(DATA_DIR, f'cleaned_train_{suffix}.pkl')
    imp_path = os.path.join(DATA_DIR, f'feature_importance_{suffix}.csv')
    
    if not os.path.exists(pkl_path) or not os.path.exists(imp_path):
        print(f'SKIP {suffix}: files not found')
        continue
    
    print(f'\n{"="*70}')
    print(f'  ABLATION: {suffix}')
    print(f'{"="*70}')
    
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    X_train = data['X']
    y_raw = data['y']
    all_classes = data['classes']
    unique_labels = sorted(np.unique(y_raw))
    label_map = {old: new for new, old in enumerate(unique_labels)}
    y_train = np.array([label_map[y] for y in y_raw])
    NUM_CLASSES = len(unique_labels)
    
    feat_imp = pd.read_csv(imp_path)
    ranked = feat_imp['Feature'].tolist()
    
    results = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for name, n_feat in FEATURE_CONFIGS:
        selected = ranked[:n_feat] if n_feat else ranked
        valid = [f for f in selected if f in X_train.columns]
        X_sub = X_train[valid]
        
        model = XGBClassifier(objective='multi:softmax', num_class=NUM_CLASSES,
            max_depth=6, learning_rate=0.3, n_estimators=100,
            eval_metric='mlogloss', random_state=42, n_jobs=-1)
        
        f1_scores = cross_val_score(model, X_sub, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
        
        # Model size & inference
        model.fit(X_sub, y_train)
        tmp = os.path.join(tempfile.gettempdir(), f'tmp_{suffix}_{name}.json')
        model.save_model(tmp)
        size_kb = os.path.getsize(tmp) / 1024
        os.remove(tmp)
        
        booster = model.get_booster()
        dm = xgb.DMatrix(X_sub.iloc[:100])
        inf_times = []
        for _ in range(10):
            t0 = time.time()
            booster.predict(dm)
            inf_times.append((time.time()-t0)*1000)
        
        r = {'Config': name, 'Features': len(valid), 'F1%': round(f1_scores.mean()*100, 2),
             'F1_std': round(f1_scores.std()*100, 2), 'Inf_ms': round(np.mean(inf_times), 1),
             'Size_KB': round(size_kb, 1)}
        results.append(r)
        print(f'  {name:8s} | {len(valid):3d} feat | F1={r["F1%"]:.2f}% | {r["Inf_ms"]:.1f}ms | {r["Size_KB"]:.0f}KB')
    
    all_ablation[suffix] = results
    del X_train, y_train, data; gc.collect()

with open(os.path.join(DATA_DIR, 'ablation_results_all.pkl'), 'wb') as f:
    pickle.dump(all_ablation, f)
print(f'\nSaved: ablation_results_all.pkl')
print('Proceed to notebook 04.')

In [ ]:
# Summary plot
fig, ax = plt.subplots(figsize=(10, 5))
for suffix, results in all_ablation.items():
    names = [r['Config'] for r in results]
    f1s = [r['F1%'] for r in results]
    ax.plot(names, f1s, marker='o', label=suffix)
ax.set_ylabel('F1-Score (macro) %')
ax.set_title('Ablation Study: F1 vs Feature Count (per sampling config)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ablation_comparison.png'), dpi=150)
plt.show()